# Homogeneous MCWF squeezing

Compare homogeneous MCWF squeezing dynamics for several atom numbers. The drive, detuning, and timestep follow the shared $N\Gamma$-scaled conventions, while the binomial sector window includes two standard deviations around $N_J=N/2$.

## Imports

In [ ]:
from pathlib import Path
import sys

import numpy as np

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from common.plotting import (
    plot_bloch_angles,
    plot_spin_components,
    plot_squeezing,
)
from common.utils.parameters import (
    mcwf_dt_from_scales,
    omega_c,
    scaled_N_Gamma,
)
from common.utils.phases import default_three_phase_protocol
from parser.mcwf import MCWFSolverParameters
from parser.moments import MomentSeries, SimulationMetadata
from post_analysis import (
    build_dressed_basis_transformation,
    compute_gell_mann_rotation,
    compute_j_moments,
    compute_mean_generator_moments,
    compute_s_moments,
    compute_squeezing,
    rotate_generator_moments,
)
from solvers.mcwf.ensamble_sim import run_trajectory_ensemble

%reload_ext autoreload
%autoreload 2

## Parameters

In [ ]:
Gamma = 1.0
N_values = (10, 50, 100)
sector_standard_deviations = 1.0
ntraj = 100
num_snapshots = 100
n_processes = -1
seed = 1234

Omega_factor = 0.1
delta_factor = 0.02
durations = (10.0, 10.0, 10.0)

# For N_J ~ Binomial(N, 1/2), sigma_NJ = sqrt(N) / 2.
dN_by_N = {
    N: int(np.ceil(sector_standard_deviations * np.sqrt(N) / 2.0))
    for N in N_values
}


# FIXME 
N_values = [100]
dN_by_N = {100:5}

print(dN_by_N)

## MCWF simulations and squeezing analysis

In [ ]:
moments_by_N = {}

for N in N_values:
    Omega0 = scaled_N_Gamma(Omega_factor, N, Gamma)
    delta0 = scaled_N_Gamma(delta_factor, N, Gamma)
    dt = mcwf_dt_from_scales(Omega0, delta0, N, Gamma)
    dN = dN_by_N[N]

    phase_protocol = default_three_phase_protocol(
        durations=durations,
        delta0=delta0,
        Omega0=Omega0,
    )
    metadata = SimulationMetadata(
        Ni=(N,),
        omega_i=(),
        Gamma=Gamma,
        phase_protocol=phase_protocol,
    )
    moments = MomentSeries(
        num_snapshots=num_snapshots,
        metadata=metadata,
    )
    parameters = MCWFSolverParameters(
        Ni=metadata.Ni,
        omega_i=metadata.omega_groups,
        Gamma=metadata.Gamma,
        phase_protocol=metadata.phase_protocol,
        dN=dN,
        sector_distribution="binomial",
        dt=dt,
        shifted_jump_operator=True,
    )

    print(
        f"N={N}: dN={dN}, Omega0={Omega0:g}, delta0={delta0:g}, "
        f"min-sector Omega0/Omega_c="
        f"{Omega0 / omega_c(N // 2 - dN, Gamma):.3f}, "
        f"dt={dt:g}"
    )
    ensemble = run_trajectory_ensemble(
        parameters,
        t_eval=moments.t,
        seed=seed,
        ntraj=ntraj,
        n_processes=n_processes,
        verbose=True,
    )

    moments.generator_moments = ensemble.moments
    moments.mean_generator_moments = compute_mean_generator_moments(
        moments.generator_moments
    )
    moments.J = compute_j_moments(
        moments.mean_generator_moments,
        metadata=moments.metadata,
    )
    moments.S = compute_s_moments(
        moments.mean_generator_moments,
        moments.J,
    )
    U = build_dressed_basis_transformation(moments.J, moments.S)
    R = compute_gell_mann_rotation(U)
    mean_dressed_generator_moments = rotate_generator_moments(
        moments.mean_generator_moments,
        R,
    )
    moments.squeezing = compute_squeezing(
        mean_dressed_generator_moments,
        moments.J,
        metadata=moments.metadata,
        include_full_system=False,
    )
    moments_by_N[N] = moments

    # MomentSeries now retains the trajectory-resolved and derived data.
    del ensemble

## Plots

In [ ]:
labels = [rf"$N={N}$" for N in N_values]
j_series = [moments_by_N[N].J for N in N_values]
squeezing_series = [
    moments_by_N[N].squeezing.group_resolved[0] for N in N_values
]
t = j_series[0].t
phase_protocol = moments_by_N[N_values[0]].metadata.phase_protocol

if any(not np.allclose(series.t, t) for series in j_series[1:]):
    raise ValueError("All runs must share the same saved-time grid.")

plot_bloch_angles(
    t,
    [series.theta[0] for series in j_series],
    [series.phi[0] for series in j_series],
    labels=labels,
    phase_protocol=phase_protocol,
    colour_family_index=None,
    shade_index=3,
    title=r"Homogeneous $J$-sphere angles",
)

plot_spin_components(
    t,
    [series.x[0] for series in j_series],
    [series.y[0] for series in j_series],
    [series.z[0] for series in j_series],
    [series.length[0] for series in j_series],
    x_sem=[None if series.x_sem is None else series.x_sem[0] for series in j_series],
    y_sem=[None if series.y_sem is None else series.y_sem[0] for series in j_series],
    z_sem=[None if series.z_sem is None else series.z_sem[0] for series in j_series],
    labels=labels,
    phase_protocol=phase_protocol,
    colour_family_index=None,
    shade_index=3,
    title=r"Homogeneous $J$-sphere spin components",
)

plot_squeezing(
    squeezing_series[0].t,
    [series.xi2 for series in squeezing_series],
    [series.lambda_min for series in squeezing_series],
    [series.N_c for series in squeezing_series],
    [series.excited_fraction_active for series in squeezing_series],
    covariance_eigvals=[
        series.covariance_eigvals for series in squeezing_series
    ],
    plot_all_eigenvalues=True,
    labels=labels,
    phase_protocol=phase_protocol,
    colour_family_index=None,
    shade_index=3,
    title="Homogeneous generalized squeezing",
)